# Modelo de Hamiltoniano XX independiente del tiempo utilizando algoritmo genético.

Partiendo del hamiltoniano XX:

$\begin{equation*}
H\ = \sum _{i=1}^{N} J_i \left( \sigma _{i}^{x} \sigma _{i+1}^{x} +\sigma _{i}^{y} \sigma _{i+1}^{y}\right)
\end{equation*}$
donde n es el largo de una cadena de espines, se busca encontrar parametros que maximicen la fidelidad en la transmisión utilizando el "algoritmo genético". Se utiliza la libreria `pygad`. Se muestran tres opciones diferentes de funciones aptitud o fitness (todas basadas en la fidelidad de la transmisiòn). Se añaden en el mismo directorio los espectros resultantes de los acoplamientos obtenidos para una cadena de n = 24 espines.

In [5]:
import numpy as np
import scipy.linalg as la
import random
import math
import csv
from random import randint
import matplotlib as mpl
import matplotlib.pyplot as plt
import pygad


In [6]:
def fidelidad(J):
    '''
    Se calcula la fidelidad de transmisión a partir de el vector de acoplamientos J.
    Para ello, se crea el hamiltoniano H, luego se diagonaliza y a partir de esto se
    calcula la proyeccion del estado inicial sobre el estado en el que la transmisión
    fue realizada. Para más información del problema físico, consultar: 
    https://doi.org/10.1007/978-3-642-39937-4
    '''

    n = 2*J.size+1
    nj = 2*J.size
    H = np.full((n, n), 0.)
    JJ = np.zeros(nj)

    for i in range(0, J.size, 1):
        JJ[i] = J[i]
        JJ[nj-i-1] = J[i]

    for i in range(0, n-1):
        H[i, i+1] = JJ[i]*(-0.5)
        H[i+1, i] = H[i, i+1]

    # Diagonalizamos este hamiltoniano
    nrows, ncol = H.shape
    (eigvals, eigvects) = la.eig(H)
    eigvals = np.real(eigvals)
    c1cn = np.full(ncol, 0.)

    for i in range(0, c1cn.size):
        c1cn[i] = np.real(eigvects[0, i])*np.real(eigvects[nrows-1, i])

    F = 0.
    Fr = 0.
    Fi = 0.

    for i in range(0, nrows):
        Fr = Fr + math.cos(eigvals[i])*c1cn[i]
        Fi = Fi + math.sin(eigvals[i])*c1cn[i]

    Fr = np.real(Fr)
    Fi = np.real(Fi)

    F = Fr*Fr+Fi*Fi

    return F


In [7]:
def fitness_func(solution, solution_idx):
    '''
    Función aptitud exactamente igual a la fidelidad
    '''
    f = fidelidad(solution)
    fitness = f
    return fitness


In [40]:
def fidelidad2(J):
    '''
    Se calcula la fidelidad de transmisión a partir de el vector de acoplamientos J.
    Pero, en este caso, se multiplica por un parámetro relacionado a la suavidad de
    los acoplamientos. Esto, es para que la función aptitud luego tenga en cuenta
    esta característica al maximizar.
    '''
    n = 2*J.size+1
    nj = 2*J.size
    H = np.full((n, n), 0.)

    JJ = np.zeros(nj)

    for i in range(0, J.size, 1):
        JJ[i] = J[i]
        JJ[nj-i-1] = J[i]

    for i in range(0, n-1):
        H[i, i+1] = JJ[i]*(-0.5)
        H[i+1, i] = H[i, i+1]

  # Diagonalizamos este hamiltoniano

    nrows, ncol = H.shape
    (eigvals, eigvects) = la.eig(H)

    eigvals = np.real(eigvals)

    c1cn = np.full(ncol, 0.)

    for i in range(0, c1cn.size):
        c1cn[i] = np.real(eigvects[0, i])*np.real(eigvects[nrows-1, i])

    F = 0.
    Fr = 0.
    Fi = 0.

    for i in range(0, nrows):
        Fr = Fr + math.cos(eigvals[i])*c1cn[i]
        Fi = Fi + math.sin(eigvals[i])*c1cn[i]

    Fr = np.real(Fr)
    Fi = np.real(Fi)

    F = Fr*Fr+Fi*Fi

    # agregamos a la fidelidad una componente asociada a la "constancia" de los acoplamientos en el centro

    Jav = np.average(JJ[2:nj//2])
    ss = 0.

    for j in range(2, n//2):
        ss = ss + (JJ[j]-Jav)**2

    Jmax = np.max(JJ[j])
    ss = ss/Jmax**2

    F = F*(math.exp(-ss))

    return F


def fitness_func2(solution, solution_idx):
    '''
    Función aptitud dependiente de la suavidad en los acoplamientos
    '''
    f = fidelidad2(solution)
    fitness = f
    return fitness


In [9]:
def fidelidad3(J):
    '''
    Se calcula la fidelidad de transmisión a partir de el vector de acoplamientos J.
    Pero, en este caso, se multiplica por un parámetro relacionado a la constancia en
    las diferencias de energía entre niveles sucesivos. Esto, se debe a la relación
    entre una buena transmisión y un espectro ordenado.
    '''
    n = 2*J.size+1
    nj = 2*J.size
    H = np.full((n, n), 0.)

    JJ = np.zeros(nj)

    for i in range(0, J.size, 1):
        JJ[i] = J[i]
        JJ[nj-i-1] = J[i]

    for i in range(0, n-1):
        H[i, i+1] = JJ[i]*(-0.5)
        H[i+1, i] = H[i, i+1]

  # --------------------------------------------
  # Diagonalizamos este hamiltoniano
  # -------------------------------------------

    nrows, ncol = H.shape
    (eigvals, eigvects) = la.eig(H)

    eigvals = np.real(eigvals)

    c1cn = np.full(ncol, 0.)

    for i in range(0, c1cn.size):
        c1cn[i] = np.real(eigvects[0, i])*np.real(eigvects[nrows-1, i])

    F = 0.
    Fr = 0.
    Fi = 0.

    for i in range(0, nrows):
        Fr = Fr + math.cos(eigvals[i])*c1cn[i]
        Fi = Fi + math.sin(eigvals[i])*c1cn[i]

    Fr = np.real(Fr)
    Fi = np.real(Fi)

    F = Fr*Fr+Fi*Fi

    # agregamos a la fidelidad una componente asociada a la "constancia" de los acoplamientos en el centro

    DeltaE = np.empty(nj)
    DeltaE_int = np.empty(nj)

    for i in range(1, nj):
        DeltaE[i] = eigvals[i]-eigvals[i-1]
        DeltaE_int[i] = int(eigvals[i]-eigvals[i-1])

    ss = 0.

    for j in range(1, nj):
        ss = ss + (DeltaE[j]-DeltaE_int[j])**2

    DEmax = np.max(DeltaE)
    ss = ss/DEmax**2

    F = F*(math.exp(-ss))

    return F


def fitness_func3(solution, solution_idx):
    'Función aptitud dependiente del espectro'
    f = fidelidad3(solution)
    fitness = f
    return fitness


In [24]:
def espectro(acop_file, en_file, nj):
    '''
    A partir de un archivo de acoplamientos (acop_file), genera un archivo con los
    valores de las energías asociadas. Debe ingresarse también la dimensión (nj) del
    sistema
    '''
    line = 0
    s = np.zeros(nj//2)

    with open(acop_file, "r") as f2:

        for x in f2:
            s[line] = x
            line = line+1

    n = 2*s.size+1
    H = np.full((n, n), 0.)
    JJ = np.zeros(nj)

    for i in range(0, s.size, 1):
        JJ[i] = s[i]
        JJ[nj-i-1] = s[i]

    for i in range(0, n-1):
        H[i, i+1] = JJ[i]*(-0.5)
        H[i+1, i] = H[i, i+1]

    # Diagonalizamos este hamiltoniano

    (eigvals, eigvects) = la.eig(H)

    with open(en_file, "w") as fw:

        writer = csv.writer(fw,  delimiter=' ')

        for j in eigvals:
            writer.writerow(np.array([np.real(j)]))

    return True


In [29]:
'''
Se establecen los parámetros del algoritmo genético,
se utiliza la primera función aptitud (idéntica a la fidelidad)
y se inicializan los genes entre 1 y n
'''

fitness_function = fitness_func
num_generations = 1000
num_parents_mating = 50
parent_selection_type = "sss"
keep_parents = 1
crossover_type = "scattered"
mutation_type = "random"
sol_per_pop = 200
nj = 24

filename = "acop_fid1_init1n.dat"  # archivo que almacena los acoplamientos

with open(filename, "w") as f2:

    writer2 = csv.writer(f2,  delimiter=' ')

    num_genes = nj//2
    init_range_low = 1
    init_range_high = nj

    ga_instance = pygad.GA(num_generations=num_generations,
                           num_parents_mating=num_parents_mating,
                           fitness_func=fitness_function,
                           sol_per_pop=sol_per_pop,
                           num_genes=num_genes,
                           init_range_low=init_range_low,
                           init_range_high=init_range_high,
                           parent_selection_type=parent_selection_type,
                           keep_parents=keep_parents,
                           crossover_type=crossover_type,
                           mutation_type=mutation_type,
                           mutation_probability=0.8,
                           stop_criteria=["reach_0.9999", "saturate_100"])

    ga_instance.run()
    solution, solution_fitness, solution_idx = ga_instance.best_solution()

    for j in solution:
        writer2.writerow(np.array([j]))

print(fidelidad(solution))  # se imprime la fidelidad máxima obtenida


0.9990555486634354


In [30]:
'''
Se establecen los parámetros del algoritmo genético,
se utiliza la primera función aptitud (idéntica a la fidelidad)
y se inicializan los genes entre 1 y 3n
'''


fitness_function = fitness_func
num_generations = 1000
num_parents_mating = 50
parent_selection_type = "sss"
keep_parents = 1
crossover_type = "scattered"
mutation_type = "random"

sol_per_pop = 200

nj = 24
filename = "n24/acop_fid1_init3n.dat"  # archivo que almacena los acoplamientos


with open(filename, "w") as f2:

    writer2 = csv.writer(f2,  delimiter=' ')

    num_genes = nj//2
    init_range_low = 1
    init_range_high = 3*nj

    ga_instance = pygad.GA(num_generations=num_generations,
                           num_parents_mating=num_parents_mating,
                           fitness_func=fitness_function,
                           sol_per_pop=sol_per_pop,
                           num_genes=num_genes,
                           init_range_low=init_range_low,
                           init_range_high=init_range_high,
                           parent_selection_type=parent_selection_type,
                           keep_parents=keep_parents,
                           crossover_type=crossover_type,
                           mutation_type=mutation_type,
                           mutation_probability=0.8,
                           stop_criteria=["reach_0.9999", "saturate_100"])

    ga_instance.run()
    solution, solution_fitness, solution_idx = ga_instance.best_solution()

    for j in solution:
        writer2.writerow(np.array([j]))
print(fidelidad(solution))  # se imprime la fidelidad máxima obtenida


0.999240893748842


In [41]:
'''
Se establecen los parámetros del algoritmo genético,
se utiliza la segunda función aptitud (dependiente de la suavidad 
en acoplamientos)
y se inicializan los genes entre 1 y 3n
'''

fitness_function = fitness_func2
num_generations = 1000
num_parents_mating = 50
parent_selection_type = "sss"
keep_parents = 1
crossover_type = "scattered"
mutation_type = "random"

sol_per_pop = 200

nj = 24
filename = "n24/acop_fid2_init3n.dat"

with open(filename, "w") as f2:

    writer2 = csv.writer(f2,  delimiter=' ')

    num_genes = nj//2
    init_range_low = 1
    init_range_high = 3*nj

    ga_instance = pygad.GA(num_generations=num_generations,
                           num_parents_mating=num_parents_mating,
                           fitness_func=fitness_function,
                           sol_per_pop=sol_per_pop,
                           num_genes=num_genes,
                           init_range_low=init_range_low,
                           init_range_high=init_range_high,
                           parent_selection_type=parent_selection_type,
                           keep_parents=keep_parents,
                           crossover_type=crossover_type,
                           mutation_type=mutation_type,
                           mutation_probability=0.8,
                           stop_criteria=["reach_0.9999", "saturate_100"])

    ga_instance.run()
    solution, solution_fitness, solution_idx = ga_instance.best_solution()

    for j in solution:
        writer2.writerow(np.array([j]))
print(fidelidad(solution))


0.9981663943132965


In [35]:
'''
Se establecen los parámetros del algoritmo genético,
se utiliza la primera función aptitud (idéntica a la fidelidad)
y se inicializan los genes entre 1 y n
'''

fitness_function = fitness_func3
num_generations = 1000
num_parents_mating = 50
parent_selection_type = "sss"
keep_parents = 1
crossover_type = "scattered"
mutation_type = "random"

sol_per_pop = 200

nj = 24
filename = "n24/acop_fid3_init3n.dat"

with open(filename, "w") as f2:

    writer2 = csv.writer(f2,  delimiter=' ')

    num_genes = nj//2
    init_range_low = 1
    init_range_high = 3*nj

    ga_instance = pygad.GA(num_generations=num_generations,
                           num_parents_mating=num_parents_mating,
                           fitness_func=fitness_function,
                           sol_per_pop=sol_per_pop,
                           num_genes=num_genes,
                           init_range_low=init_range_low,
                           init_range_high=init_range_high,
                           parent_selection_type=parent_selection_type,
                           keep_parents=keep_parents,
                           crossover_type=crossover_type,
                           mutation_type=mutation_type,
                           mutation_probability=0.8,
                           stop_criteria=["reach_0.9999", "saturate_100"])

    ga_instance.run()
    solution, solution_fitness, solution_idx = ga_instance.best_solution()

    for j in solution:
        writer2.writerow(np.array([j]))
        
print(fidelidad(solution))


0.9986388430143821


Se verifica que con las tres funciones se logra una transmisión con fidelidad de 99.9%. A continuación, se generan los archivos que contienen los espectros, utilizados para obtener la figura presente en el directorio.

In [25]:
espectro('n24/acop_fid3_init3n.dat', 'n24/en_fid3_init3n.dat', nj)


True

In [42]:
espectro('n24/acop_fid2_init3n.dat', 'n24/en_fid2_init3n.dat', nj)


True

In [27]:
espectro('n24/acop_fid1_init3n.dat', 'n24/en_fid1_init3n.dat', nj)


True

In [28]:
espectro('n24/acop_fid1_init1n.dat', 'n24/en_fid1_init1n.dat', nj)


True